local_cache_dir

(1) check_src:
    if not in local_dir, check if exists and pull first
    check size and digest -> continue / abort

(2) if not partial:
    copy! -> shutil.copyobject
    check size and digest -> continue / abort

(3) if partial:
    check_dst: size and digest -> continue / abort
    append! -> shutil.copyobject
    check size and digest -> continue / abort


In [2]:
import os
from tqdm import tqdm
from io import BufferedReader, BufferedWriter

def copyfileobj_progress(
    fsrc: BufferedReader,
    fdst: BufferedWriter,
    buffer_size: int = 64 * 1024 if os.name != 'nt' else 1024 * 1024,
    desc: str = 'copy',
):
    """
    Copy data from file-like object fsrc to file-like object fdst,
    behaves the same as shutil.copyfileobj but with progress bar.

    ref: python3.10/shutil.py#L187
    """
    src_size = os.fstat(fsrc.fileno()).st_size
    fsrc_read = fsrc.read
    fdst_write = fdst.write

    with tqdm(total=src_size, unit='B', unit_scale=True, desc=desc) as pbar:
        while True:
            buf = fsrc_read(buffer_size)
            if not buf:
                break
            fdst_write(buf)
            pbar.update(len(buf))

with open('wocsync.tasks.jsonl', 'rb') as fsrc, open('wocsync.tasks.jsonl.copy', 'wb') as fdst:
    copyfileobj_progress(fsrc, fdst)

copy: 100%|██████████| 408k/408k [00:00<00:00, 49.0MB/s]


In [4]:
import os 

def _append_file(src, dst):
    with open(src, 'rb') as src_file, open(dst, 'ab') as dst_file:
        copyfileobj_progress(src_file, dst_file, desc=os.path.basename(dst))

_append_file('wocsync.tasks.jsonl', 'wocsync.tasks.jsonl.copy')

wocsync.tasks.jsonl.copy: 100%|██████████| 408k/408k [00:00<00:00, 99.1MB/s]


In [5]:
from syncmate.base import WocSyncCopyTask, WocSyncPartialCopyTask, deserialize_tasks

_tasks = deserialize_tasks('wocsync.tasks.jsonl')
_tasks

[WocSyncCopyTask(src_path='/da4_fast/b2PFullV.0.tch', dst_path='/woc_data/basemaps/b2PFullV.0.tch', size=63309884528, digest='e654ebc75e2ae967'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.1.tch', dst_path='/woc_data/basemaps/b2PFullV.1.tch', size=63638552608, digest='81b5320ba40e0974'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.10.tch', dst_path='/woc_data/basemaps/b2PFullV.10.tch', size=63993858336, digest='a8de63b925b1a0d7'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.11.tch', dst_path='/woc_data/basemaps/b2PFullV.11.tch', size=63657506976, digest='e0643df2876dd5f7'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.12.tch', dst_path='/woc_data/basemaps/b2PFullV.12.tch', size=63688352080, digest='906475420f6eac1a'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.13.tch', dst_path='/woc_data/basemaps/b2PFullV.13.tch', size=63711762576, digest='be0694040eb5364b'),
 WocSyncCopyTask(src_path='/da4_fast/b2PFullV.14.tch', dst_path='/woc_data/basemaps/b2PFullV.14.tch', size=634700544

In [11]:
sum(x.size for x in filter(lambda x: 'blob' in x.src_path and 'bin' in x.src_path, _tasks))

290309582039028

In [13]:
WOCSYNC_LOCAL_CACHE = '/archive/woc/parts/'

# list dir

import os
for f in os.listdir(WOCSYNC_LOCAL_CACHE):
    print(f)

tree_20.bin.part.193935052135.11df052324539e2e
commit_65.idx.part.415871719.cc505ac602c28ec2
tree_121.idx.part.5313707797.873b476cb8f42f10
commit_123.idx.part.415814925.f9ef1c7120ec2e93
commit_106.idx.part.416027209.fcd11062ea56ae52
commit_99.idx.part.415919620.67bf17e6c2bfec8c
commit_35.bin.part.2267666284.cdb7efed14ed17cb
tree_29.idx.part.5313167548.4ae14730aaefae3a
commit_38.idx.part.415862178.0fe77b6430d81a08
commit_3.bin.part.2272601602.94cbe092da260170
tree_7.idx.part.5311190433.a822c53dedc81d69
commit_30.idx.part.416078656.4d36881cad6702f1
commit_117.bin.part.2262501331.ee8da0e5d53d2fb1
tree_92.idx.part.5312249622.aebba6426635756c
tree_11.idx.part.5312372085.a6dac69889e275c4
tree_1.idx.part.5311582280.58bef57888eec4f2
blob_17.idx.part.6322832242.b02c2ea7301a7495
tree_116.idx.part.5312590648.e2a983fa5ae65b0b
commit_99.bin.part.2307741001.189d9df856faad51
tree_114.idx.part.5312903541.f0c067d52aa09c10
blob_119.idx.part.6322114704.ee68ffc98e4c02f8
commit_50.idx.part.415976666.6c9ca0

In [16]:
from woc.utils import sample_md5

def _validate_part(
    file_path: str
):
    assert os.path.isfile(file_path), f'{file_path}: not found'
    expected_size, expected_md5 = int(file_path.split('.')[-2]), file_path.split('.')[-1]
    actual_size = os.path.getsize(file_path)
    assert actual_size == expected_size, f'{file_path}: size mismatch, expected {expected_size}, got {actual_size}'
    actual_md5 = sample_md5(file_path)
    assert actual_md5 == expected_md5, f'{file_path}: md5 mismatch, expected {expected_md5}, got {actual_md5}'

_validate_part(os.path.join(WOCSYNC_LOCAL_CACHE, 'blob_22.bin.part.20832…43564.9848764e43f9e2f9'))

AssertionError: /archive/woc/parts/blob_22.bin.part.20832…43564.9848764e43f9e2f9: not found

In [7]:
def get_remote_fname(
    bucket: str,
    task: WocSyncCopyTask,
):
    if isinstance(task, WocSyncPartialCopyTask):
        return f"{bucket}/{os.path.basename(task.src_path)}.part.{task.size}.{task.part_digest}"
    return f"{bucket}/{os.path.basename(task.src_path)}"

In [40]:
import logging 

def append_part(
    task: WocSyncPartialCopyTask,
):
    _local_path = os.path.join(WOCSYNC_LOCAL_CACHE, f"{os.path.basename(task.src_path)}.part.{task.size}.{task.part_digest}")
    print(f'append_part: {_local_path} -> {task.dst_path}')
    if not os.path.isfile(_local_path):
        print(f'{_local_path}: not found')
        return
    
    # check the size of the original
    assert os.path.isfile(task.dst_path), f'{task.dst_path}: not found'
    original_size = os.path.getsize(task.dst_path)
    if original_size == task.size + task.skip and sample_md5(task.dst_path) == task.digest:
        print(f'{task.dst_path}: already complete')
        return
    
    # validate part
    assert os.path.isfile(_local_path), f'{_local_path}: not found'
    expected_size, expected_md5 = int(_local_path.split('.')[-2]), _local_path.split('.')[-1]
    part_size = os.path.getsize(_local_path)
    assert part_size == expected_size, f'{_local_path}: size mismatch, expected {expected_size}, got {part_size}'
    part_md5 = sample_md5(_local_path)
    assert part_md5 == expected_md5, f'{_local_path}: md5 mismatch, expected {expected_md5}, got {part_md5}'
    
    if original_size == task.skip: # append
        with open(_local_path, 'rb') as src_file, open(task.dst_path, 'ab') as dst_file:
            copyfileobj_progress(src_file, dst_file, desc=os.path.basename(task.dst_path))
    else:  # in case of interruption, seek and copy
        logging.warning(f'{task.dst_path}: size mismatch, expected {task.skip}, got {original_size}')
        # first seek the dst file
        assert task.skip <= original_size, f'{task.dst_path}: skip too large, expected <= {original_size}, got {task.skip}'
        with open(_local_path, 'rb') as src_file, open(task.dst_path, 'r+b') as dst_file:
            dst_file.seek(task.skip)
            copyfileobj_progress(src_file, dst_file, desc=os.path.basename(task.dst_path))

    # check the size of the combined file
    full_size = os.path.getsize(task.dst_path)
    assert full_size == task.size + task.skip, f'{task.dst_path}: size mismatch, expected {task.size + task.skip}, got {full_size}'
    full_md5 = sample_md5(task.dst_path)
    assert full_md5 == task.digest, f'{task.dst_path}: md5 mismatch, expected {task.digest}, got {full_md5}'

In [31]:
_partial_tasks = list(filter(lambda x: isinstance(x, WocSyncPartialCopyTask), _tasks))
_partial_task = _partial_tasks[2]
_partial_task

WocSyncPartialCopyTask(src_path='/da5_data/All.blobs/commit_2.idx', dst_path='/woc_data/All.blobs/commit_2.idx', size=415869216, digest='5ef259f6ca59068d', skip=1980533229, part_digest='6c806407d3c70696')

In [39]:
from tqdm import tqdm
import logging

for task in tqdm(_partial_tasks):
    try:
        append_part(task)
    except AssertionError as e:
        logging.error(e)

  0%|          | 0/761 [00:00<?, ?it/s]WARNING:root:/woc_data/All.blobs/commit_97.idx: size mismatch, expected 1980252485, got 0
ERROR:root:/woc_data/All.blobs/commit_97.idx: skip too large, expected <= 0, got 1980252485
ERROR:root:/woc_data/All.blobs/commit_98.idx: skip too large, expected <= 0, got 1979652035
 13%|█▎        | 99/761 [00:00<00:00, 988.15it/s]WARNING:root:/woc_data/All.blobs/commit_99.idx: size mismatch, expected 1980022365, got 0
ERROR:root:/woc_data/All.blobs/commit_99.idx: skip too large, expected <= 0, got 1980022365
ERROR:root:/woc_data/All.blobs/commit_100.idx: skip too large, expected <= 0, got 1979677716
ERROR:root:/woc_data/All.blobs/commit_101.idx: skip too large, expected <= 0, got 1979425392
ERROR:root:/woc_data/All.blobs/commit_102.idx: skip too large, expected <= 0, got 1979604031
ERROR:root:/woc_data/All.blobs/commit_103.idx: skip too large, expected <= 0, got 1980327449
ERROR:root:/woc_data/All.blobs/commit_104.idx: skip too large, expected <= 0, got 19

append_part: /archive/woc/parts/commit_0.idx.part.416148851.01c47d9ed30f8e00 -> /woc_data/All.blobs/commit_0.idx
/woc_data/All.blobs/commit_0.idx: already complete
append_part: /archive/woc/parts/commit_1.idx.part.415972557.40c05b2c6b144a43 -> /woc_data/All.blobs/commit_1.idx
/woc_data/All.blobs/commit_1.idx: already complete
append_part: /archive/woc/parts/commit_2.idx.part.415869216.6c806407d3c70696 -> /woc_data/All.blobs/commit_2.idx
/woc_data/All.blobs/commit_2.idx: already complete
append_part: /archive/woc/parts/commit_3.idx.part.415804610.31c3524871f06967 -> /woc_data/All.blobs/commit_3.idx
/woc_data/All.blobs/commit_3.idx: already complete
append_part: /archive/woc/parts/commit_4.idx.part.415972011.4627fb8754b705a1 -> /woc_data/All.blobs/commit_4.idx
/woc_data/All.blobs/commit_4.idx: already complete
append_part: /archive/woc/parts/commit_5.idx.part.416053355.a765a90e1c6e0a6f -> /woc_data/All.blobs/commit_5.idx
/woc_data/All.blobs/commit_5.idx: already complete
append_part: /ar

ERROR:root:/woc_data/All.blobs/commit_41.bin: skip too large, expected <= 0, got 9755126279
ERROR:root:/woc_data/All.blobs/commit_42.bin: skip too large, expected <= 0, got 9731315195
ERROR:root:/woc_data/All.blobs/commit_43.bin: skip too large, expected <= 0, got 9768680250
ERROR:root:/woc_data/All.blobs/commit_44.bin: skip too large, expected <= 0, got 9786549213
ERROR:root:/woc_data/All.blobs/commit_45.bin: skip too large, expected <= 0, got 9742130015
ERROR:root:/woc_data/All.blobs/commit_46.bin: skip too large, expected <= 0, got 9735091131
ERROR:root:/woc_data/All.blobs/commit_47.bin: skip too large, expected <= 0, got 9741864444
ERROR:root:/woc_data/All.blobs/commit_48.bin: skip too large, expected <= 0, got 9752491830
ERROR:root:/woc_data/All.blobs/commit_49.bin: skip too large, expected <= 0, got 9753221705
ERROR:root:/woc_data/All.blobs/commit_50.bin: skip too large, expected <= 0, got 9761177748
ERROR:root:/woc_data/All.blobs/commit_51.bin: skip too large, expected <= 0, got

append_part: /archive/woc/parts/commit_42.bin.part.2264850898.289e48ad3e5616b5 -> /woc_data/All.blobs/commit_42.bin
append_part: /archive/woc/parts/commit_43.bin.part.2259373099.d8695d461b9e4d37 -> /woc_data/All.blobs/commit_43.bin
append_part: /archive/woc/parts/commit_44.bin.part.2259853158.d8673ef44336ca54 -> /woc_data/All.blobs/commit_44.bin
append_part: /archive/woc/parts/commit_45.bin.part.2259669433.da162961b96d57f0 -> /woc_data/All.blobs/commit_45.bin
append_part: /archive/woc/parts/commit_46.bin.part.2281440243.bad06cf7c40f9edd -> /woc_data/All.blobs/commit_46.bin
append_part: /archive/woc/parts/commit_47.bin.part.2300362923.aca7a3417a6b1ea0 -> /woc_data/All.blobs/commit_47.bin
append_part: /archive/woc/parts/commit_48.bin.part.2257578263.b89e714db0d17c7e -> /woc_data/All.blobs/commit_48.bin
append_part: /archive/woc/parts/commit_49.bin.part.2258771980.c8d9016f2e21e61e -> /woc_data/All.blobs/commit_49.bin
append_part: /archive/woc/parts/commit_50.bin.part.2259674649.dd534346db

commit_80.bin: 100%|██████████| 2.26G/2.26G [00:15<00:00, 150MB/s]


append_part: /archive/woc/parts/commit_81.bin.part.2272804888.e90b86e0c14ae9e2 -> /woc_data/All.blobs/commit_81.bin


commit_81.bin: 100%|██████████| 2.27G/2.27G [00:15<00:00, 143MB/s] 
 27%|██▋       | 208/761 [00:31<02:16,  4.04it/s] 

append_part: /archive/woc/parts/commit_82.bin.part.2257751421.f3244235784b0f3d -> /woc_data/All.blobs/commit_82.bin


commit_82.bin:   9%|▊         | 194M/2.26G [00:01<00:15, 133MB/s]
 27%|██▋       | 208/761 [00:33<01:28,  6.24it/s]


KeyboardInterrupt: 

In [34]:
append_part(_partial_task)

append_part: /archive/woc/parts/commit_2.idx.part.415869216.6c806407d3c70696 -> /woc_data/All.blobs/commit_2.idx


AssertionError: /woc_data/All.blobs/commit_2.idx: size mismatch, expected 1980533229, got 2396402445

In [29]:
full_md5 = sample_md5(_partial_task.dst_path)
assert full_md5 == _partial_task.digest, f'{_partial_task.dst_path}: md5 mismatch, expected {_partial_task.digest}, got {full_md5}'

In [ ]:
def _sync_file(task: WocSyncCopyTask):
    _remote = get_remote_fname('r2:woc', task)